In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.data.market_loader import MarketLoader
from src.data.synthetic_sofr_builder import build_term_sofr_curve

from src.term_structure.bootstrap_curve import (
    bootstrap_df_from_sofr, 
    build_zero_curve_from_df,
    bootstrap_df_from_treasury_curve
)
from src.term_structure.curve_interpolation import (
    build_coupon_structure,
    zero_rate_to_df,
    interpolate_zero_curve
)

In [2]:
# downloading market curves
loader = MarketLoader()
curves = loader.loader_pipeline()

# building synthetic sofr curve from ON rates and futures
sofr_curve = build_term_sofr_curve(curves = curves)
sofr_curve

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..


,ON,3M,6M,1M
Date,,,,
2018-04-03,1.83,1.72,1.88,1.79
2018-04-04,1.74,1.68,1.86,1.72
2018-04-05,1.75,1.69,1.88,1.73
2018-04-06,1.75,1.70,1.86,1.73
2018-04-09,1.75,1.73,1.89,1.74
...,...,...,...,...
2026-04-15,3.72,3.62,3.60,3.69
2026-04-16,3.67,3.62,3.58,3.65
2026-04-17,3.65,3.61,3.56,3.64


In [3]:
# bootstrapping discount factors from sofr curve
df_bootstrap_from_sofr = bootstrap_df_from_sofr(
    sofr_curve = sofr_curve
)

df_bootstrap_from_sofr

,ON,3M,6M,1M
2018-04-03,0.999949,0.995718,0.990688,0.998511
2018-04-04,0.999952,0.995818,0.990786,0.998569
2018-04-05,0.999951,0.995793,0.990688,0.998560
2018-04-06,0.999951,0.995768,0.990786,0.998560
2018-04-09,0.999951,0.995694,0.990638,0.998552
...,...,...,...,...
2026-04-15,0.999897,0.991031,0.982318,0.996934
2026-04-16,0.999898,0.991031,0.982415,0.996968
2026-04-17,0.999899,0.991056,0.982511,0.996976
2026-04-20,0.999899,0.991031,0.982367,0.996984


In [4]:
# constructing zero curve from discount curve
zero_curve_mm = build_zero_curve_from_df(
    discount_curve = df_bootstrap_from_sofr
)

zero_curve_mm

,ON,3M,6M,1M
2018-04-03,1.829953,1.716313,1.871219,1.788666
2018-04-04,1.739958,1.676482,1.851404,1.718769
2018-04-05,1.749957,1.686440,1.871219,1.728754
2018-04-06,1.749957,1.696398,1.851404,1.728754
2018-04-09,1.749957,1.726270,1.881126,1.738740
...,...,...,...,...
2026-04-15,3.719808,3.603718,3.567984,3.684338
2026-04-16,3.669813,3.603718,3.548336,3.644460
2026-04-17,3.649815,3.593807,3.528687,3.634490
2026-04-20,3.629817,3.603718,3.558160,3.624521


In [5]:
# interpolate zero curve to coupon structure
coupon_structure = build_coupon_structure(
    max_year = 30,
    freq = 2
)

zero_curve_mm_interp = interpolate_zero_curve(
    zero_curve_df = zero_curve_mm,
    target_times = coupon_structure
)

# df bootstrap using interpolated zero curve
df_mm_interp = zero_rate_to_df(
    zero_curve_interp = zero_curve_mm_interp
)

df_mm_interp

,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0,...,25.5,26.0,26.5,27.0,27.5,28.0,28.5,29.0,29.5,30.0
2018-04-03,0.990688,0.981462,0.972322,0.963267,0.954297,0.945410,0.936606,0.927884,0.919243,0.910683,...,0.620543,0.614764,0.609039,0.603367,0.597748,0.592182,0.586667,0.581204,0.575792,0.570429
2018-04-04,0.990786,0.981656,0.972611,0.963649,0.954770,0.945972,0.937256,0.928620,0.920063,0.911585,...,0.623686,0.617939,0.612245,0.606604,0.601015,0.595477,0.589990,0.584553,0.579167,0.573830
2018-04-05,0.990688,0.981462,0.972322,0.963267,0.954297,0.945410,0.936606,0.927884,0.919243,0.910683,...,0.620543,0.614764,0.609039,0.603367,0.597748,0.592182,0.586667,0.581204,0.575792,0.570429
2018-04-06,0.990786,0.981656,0.972611,0.963649,0.954770,0.945972,0.937256,0.928620,0.920063,0.911585,...,0.623686,0.617939,0.612245,0.606604,0.601015,0.595477,0.589990,0.584553,0.579167,0.573830
2018-04-09,0.990638,0.981365,0.972177,0.963076,0.954061,0.945129,0.936281,0.927516,0.918833,0.910232,...,0.618977,0.613183,0.607442,0.601756,0.596122,0.590542,0.585013,0.579537,0.574111,0.568737
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,0.982318,0.964949,0.947887,0.931127,0.914663,0.898490,0.882603,0.866997,0.851667,0.836608,...,0.402590,0.395472,0.388479,0.381610,0.374863,0.368234,0.361723,0.355328,0.349045,0.342873
2026-04-16,0.982415,0.965139,0.948167,0.931493,0.915112,0.899020,0.883210,0.867679,0.852421,0.837431,...,0.404612,0.397497,0.390507,0.383640,0.376894,0.370266,0.363755,0.357358,0.351074,0.344900
2026-04-17,0.982511,0.965328,0.948446,0.931859,0.915562,0.899550,0.883818,0.868361,0.853175,0.838254,...,0.406645,0.399533,0.392546,0.385681,0.378936,0.372309,0.365797,0.359400,0.353115,0.346939
2026-04-20,0.982367,0.965044,0.948027,0.931310,0.914888,0.898755,0.882907,0.867338,0.852044,0.837019,...,0.403600,0.396483,0.389492,0.382624,0.375877,0.369249,0.362738,0.356341,0.350058,0.343885


In [6]:
### constructing discount factors from treasury par yields
# get treasury par yields
treasury_curve = curves['treasury']

# bootstrapping discount factors from treasury par yields merging money market dfs
df_bootstrap_from_treasury_full = bootstrap_df_from_treasury_curve(
    treasury_curve = treasury_curve,
    interpolated_dfs = df_mm_interp
)

df_bootstrap_from_treasury_full

,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0,...,25.5,26.0,26.5,27.0,27.5,28.0,28.5,29.0,29.5,30.0
2018-04-03,0.990688,0.979412,0.972322,0.955563,0.954297,0.945410,0.936606,0.927884,0.919243,0.877040,...,0.620543,0.614764,0.609039,0.603367,0.597748,0.592182,0.586667,0.581204,0.575792,0.315669
2018-04-04,0.990786,0.979606,0.972611,0.955556,0.954770,0.945972,0.937256,0.928620,0.920063,0.876524,...,0.623686,0.617939,0.612245,0.606604,0.601015,0.595477,0.589990,0.584553,0.579167,0.311666
2018-04-05,0.990688,0.979607,0.972322,0.955175,0.954297,0.945410,0.936606,0.927884,0.919243,0.875175,...,0.620543,0.614764,0.609039,0.603367,0.597748,0.592182,0.586667,0.581204,0.575792,0.304594
2018-04-06,0.990786,0.979704,0.972611,0.955748,0.954770,0.945972,0.937256,0.928620,0.920063,0.877921,...,0.623686,0.617939,0.612245,0.606604,0.601015,0.595477,0.589990,0.584553,0.579167,0.316091
2018-04-09,0.990638,0.979510,0.972177,0.955371,0.954061,0.945129,0.936281,0.927516,0.918833,0.877064,...,0.618977,0.613183,0.607442,0.601756,0.596122,0.590542,0.585013,0.579537,0.574111,0.316539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,0.982318,0.963993,0.947887,0.928140,0.914663,0.898490,0.882603,0.866997,0.851667,0.823328,...,0.402590,0.395472,0.388479,0.381610,0.374863,0.368234,0.361723,0.355328,0.349045,0.114769
2026-04-16,0.982415,0.964087,0.948167,0.927756,0.915112,0.899020,0.883210,0.867679,0.852421,0.822824,...,0.404612,0.397497,0.390507,0.383640,0.376894,0.370266,0.363755,0.357358,0.351074,0.105719
2026-04-17,0.982511,0.964563,0.948446,0.929054,0.915562,0.899550,0.883818,0.868361,0.853175,0.825839,...,0.406645,0.399533,0.392546,0.385681,0.378936,0.372309,0.365797,0.359400,0.353115,0.112297
2026-04-20,0.982367,0.964470,0.948027,0.928878,0.914888,0.898755,0.882907,0.867338,0.852044,0.825051,...,0.403600,0.396483,0.389492,0.382624,0.375877,0.369249,0.362738,0.356341,0.350058,0.115390
